# Basic Movie Recommender System

**Objective:** Recommend similar movies from audience-rating patterns using item-to-item collaborative similarity.

The project uses the MovieLens-style files already committed in this folder and applies a minimum-support rule so correlations are not dominated by one-off ratings.


## 1. Setup and portable paths


In [1]:
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="deep")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__} | NumPy: {np.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Random seed: {RANDOM_STATE}")

from pathlib import Path

try:
    PROJECT_DIR = Path(__file__).resolve().parent
except NameError:
    PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "Review.data").exists():
    PROJECT_DIR = Path.cwd() / "Unsupervised Learning Projects" / "Basic Movie Recommender System"
print(f"Resolved project folder: {PROJECT_DIR.name}")


Python: 3.12.13
pandas: 2.2.3 | NumPy: 2.3.5 | scikit-learn: 1.8.0
Random seed: 42
Resolved project folder: Basic Movie Recommender System


## 2. Load and validate the included ratings


In [2]:
ratings = pd.read_csv(PROJECT_DIR / "Review.data", sep="\t", names=["user_id", "movie_id", "rating", "timestamp"])
titles = pd.read_csv(PROJECT_DIR / "Movie_Id_Titles_data")
titles.columns = ["movie_id", "title"]
data = ratings.merge(titles, on="movie_id", how="inner")
print(f"Ratings: {len(data):,} | users: {data['user_id'].nunique():,} | movies: {data['movie_id'].nunique():,}")
display(data.head())
display(data["rating"].describe().round(3).to_frame())


Ratings: 100,003 | users: 944 | movies: 1,682


,user_id,movie_id,rating,timestamp,title
0,0,50,5,881250949,Star Wars (1977)
1,0,172,5,881250949,"Empire Strikes Back, The (1980)"
2,0,133,1,881250949,Gone with the Wind (1939)
3,196,242,3,881250949,Kolya (1996)
4,186,302,3,891717742,L.A. Confidential (1997)


,rating
count,100003.000
mean,3.530
std,1.126
min,1.000
25%,3.000
50%,4.000
75%,4.000
max,5.000


## 3. Popularity and sparsity diagnostics


In [3]:
movie_stats = data.groupby("title").agg(mean_rating=("rating", "mean"), rating_count=("rating", "size")).sort_values("rating_count", ascending=False)
display(movie_stats.head(12).round(3))
matrix = data.pivot_table(index="user_id", columns="title", values="rating")
sparsity = 1 - matrix.notna().sum().sum() / matrix.size
print(f"User-item matrix: {matrix.shape[0]} × {matrix.shape[1]} | sparsity: {sparsity:.2%}")


User-item matrix: 944 × 1664 | sparsity: 93.65%


,mean_rating,rating_count
title,,
Star Wars (1977),4.360,584
Contact (1997),3.804,509
Fargo (1996),4.156,508
Return of the Jedi (1983),4.008,507
Liar Liar (1997),3.157,485
"English Patient, The (1996)",3.657,481
Scream (1996),3.441,478
Toy Story (1995),3.878,452
Air Force One (1997),3.631,431


## 4. Item-based collaborative similarity


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

MIN_RATINGS = 40
eligible = movie_stats.index[movie_stats["rating_count"] >= MIN_RATINGS]
centered = matrix[eligible].sub(matrix[eligible].mean(axis=1), axis=0).fillna(0)
similarity = cosine_similarity(centered.T)
similarity_df = pd.DataFrame(similarity, index=eligible, columns=eligible)
print(f"Eligible movies with at least {MIN_RATINGS} ratings: {len(eligible)}")

def recommend(title, top_n=8):
    if title not in similarity_df:
        raise KeyError(f"{title!r} is not an eligible title")
    scores = similarity_df[title].drop(title).sort_values(ascending=False).head(top_n)
    return movie_stats.loc[scores.index].assign(similarity=scores).sort_values("similarity", ascending=False)

query_title = "Star Wars (1977)" if "Star Wars (1977)" in similarity_df else similarity_df.index[0]
print(f"Recommendations for: {query_title}")
display(recommend(query_title).round(4))


Eligible movies with at least 40 ratings: 708
Recommendations for: Star Wars (1977)


,mean_rating,rating_count,similarity
title,,,
Return of the Jedi (1983),4.0079,507,0.6450
"Empire Strikes Back, The (1980)",4.2065,368,0.6232
Raiders of the Lost Ark (1981),4.2524,420,0.5407
"Godfather, The (1972)",4.2833,413,0.3582
"Princess Bride, The (1987)",4.1728,324,0.3402
"Usual Suspects, The (1995)",4.3858,267,0.3318
"Shawshank Redemption, The (1994)",4.4452,283,0.3111
Blade Runner (1982),4.1382,275,0.3028


## 5. Qualitative checks across multiple titles


In [5]:
queries = [title for title in ["Star Wars (1977)", "Liar Liar (1997)", "Toy Story (1995)"] if title in similarity_df]
for title in queries:
    print(f"\nTop neighbours for {title}")
    display(recommend(title, top_n=5).round(4))



Top neighbours for Star Wars (1977)

Top neighbours for Liar Liar (1997)

Top neighbours for Toy Story (1995)


,mean_rating,rating_count,similarity
title,,,
Return of the Jedi (1983),4.0079,507,0.6450
"Empire Strikes Back, The (1980)",4.2065,368,0.6232
Raiders of the Lost Ark (1981),4.2524,420,0.5407
"Godfather, The (1972)",4.2833,413,0.3582
"Princess Bride, The (1987)",4.1728,324,0.3402


,mean_rating,rating_count,similarity
title,,,
George of the Jungle (1997),2.6852,162,0.1915
Volcano (1997),2.8082,219,0.1425
Waterworld (1995),2.8039,102,0.1423
Booty Call (1997),2.1667,48,0.1420
Leave It to Beaver (1997),1.8409,44,0.1412


,mean_rating,rating_count,similarity
title,,,
Raiders of the Lost Ark (1981),4.2524,420,0.2412
Beauty and the Beast (1991),3.7921,202,0.2144
Apollo 13 (1995),3.9312,276,0.2005
Star Wars (1977),4.3596,584,0.2000
Aladdin (1992),3.8128,219,0.1834


## 6. Findings and limitations

- Minimum rating support reduces unstable one-user correlations.
- Collaborative similarity cannot solve cold-start for unseen movies or users.
- Similarity is an offline discovery signal, not evidence of user satisfaction.
- A production system should use time-aware offline evaluation and online experiments.
